In [10]:
import pandas as pd
import requests
from io import StringIO  # <-- 1. Adicione esta importação

def coletar_dados_fundamentus():
    # Disfarçamos nosso script como um navegador comum
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    url = 'https://www.fundamentus.com.br/resultado.php'
    
    resposta = requests.get(url, headers=headers)
    
    # 2. O truque está aqui: StringIO engana o Pandas, fazendo-o pensar que o texto é um arquivo
    tabelas = pd.read_html(StringIO(resposta.text), thousands='.', decimal=',')
    df = tabelas[0]
    
    # Limpeza básica: removendo o símbolo '%' e convertendo para número
    colunas_porcentagem = ['Div.Yield', 'Mrg Ebit', 'Mrg. Líq.', 'ROIC', 'ROE', 'Cresc. Rec.5a']
    for col in colunas_porcentagem:
        df[col] = df[col].astype(str).str.replace('%', '').str.replace('.', '').str.replace(',', '.').astype(float) / 100
    
    return df

dados_acoes = coletar_dados_fundamentus()
print(f"Coletados dados de {len(dados_acoes)} ativos.")



Coletados dados de 994 ativos.


In [11]:
from sqlalchemy import create_engine

# Cria um banco de dados local chamado 'mercado_financeiro.db'
engine = create_engine('sqlite:///mercado_financeiro.db')

# Salva o DataFrame no banco. Se a tabela já existir, ele substitui (replace).
# Em um ambiente real, você faria um "append" ou "upsert", mas para a foto atual, "replace" funciona.
dados_acoes.to_sql('indicadores_acoes', con=engine, if_exists='replace', index=False)

print("Dados salvos com sucesso no banco de dados SQLite!")

Dados salvos com sucesso no banco de dados SQLite!


In [12]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Conectar no banco e puxar os dados
engine = create_engine('sqlite:///mercado_financeiro.db')
df = pd.read_sql('indicadores_acoes', con=engine)

# 2. Filtrar dados limpos (remover ações com volume zerado ou P/L negativo)
df_limpo = df[(df['Liq.2meses'] > 1000000) & (df['P/L'] > 0)].dropna()

# 3. Escolher os fatores (Features) para o algoritmo
# Vamos buscar Dividendos, Preço/Lucro (Valuation) e ROE (Qualidade)
features = ['Div.Yield', 'P/L', 'ROE']
X = df_limpo[features]

# 4. Padronização (Crucial para o K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Aplicar o K-Means pedindo 3 grupos (Clusters)
kmeans = KMeans(n_clusters=3, random_state=42)
df_limpo['Perfil_Risco_Cluster'] = kmeans.fit_predict(X_scaled)

# 6. Salvar o resultado de volta no banco em uma nova tabela para o Power BI
df_limpo.to_sql('ativos_clusterizados', con=engine, if_exists='replace', index=False)
print("Ativos classificados e salvos no banco!")

Ativos classificados e salvos no banco!


In [18]:
import sqlite3

# 1. Conectar ao banco de dados
# Lembre-se de colocar o caminho correto onde está o seu mercado_financeiro.db
con = sqlite3.connect(r'mercado_financeiro.db')

# 2. Carregar a tabela final que vai para o Power BI
df = pd.read_sql_query("SELECT * from ativos_clusterizados", con)

# --- O DIAGNÓSTICO ---
print("📊 TIPOS DE DADOS ANTES DA LIMPEZA:")
print(df.dtypes)
print("-" * 50)



📊 TIPOS DE DADOS ANTES DA LIMPEZA:
Papel                       str
Cotação                 float64
P/L                     float64
P/VP                    float64
PSR                     float64
Div.Yield               float64
P/Ativo                 float64
P/Cap.Giro              float64
P/EBIT                  float64
P/Ativ Circ.Liq         float64
EV/EBIT                 float64
EV/EBITDA               float64
Mrg Bruta                   str
Mrg Ebit                float64
Mrg. Líq.               float64
Liq. Corr.              float64
ROIC                    float64
ROE                     float64
Liq.2meses              float64
Patrim. Líq             float64
Dív.Líq/ Patrim.            str
Cresc. Rec.5a           float64
Perfil_Risco_Cluster      int64
dtype: object
--------------------------------------------------


In [19]:
# 3. Limpeza dos dados: convertendo texto para número
colunas_para_corrigir = ['Mrg Bruta', 'Dív.Líq/ Patrim.']

for col in colunas_para_corrigir:
    # Transforma em string para garantir, remove o '%' e troca a ',' por '.'
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = df[col].str.replace(',', '.', regex=False)
    
    # Força a conversão para número (se encontrar algum erro/texto estranho, vira NaN/Nulo)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- O DIAGNÓSTICO DEPOIS DA LIMPEZA ---
print("\n✅ TIPOS DE DADOS DEPOIS DA LIMPEZA:")
print(df.dtypes)


✅ TIPOS DE DADOS DEPOIS DA LIMPEZA:
Papel                       str
Cotação                 float64
P/L                     float64
P/VP                    float64
PSR                     float64
Div.Yield               float64
P/Ativo                 float64
P/Cap.Giro              float64
P/EBIT                  float64
P/Ativ Circ.Liq         float64
EV/EBIT                 float64
EV/EBITDA               float64
Mrg Bruta               float64
Mrg Ebit                float64
Mrg. Líq.               float64
Liq. Corr.              float64
ROIC                    float64
ROE                     float64
Liq.2meses              float64
Patrim. Líq             float64
Dív.Líq/ Patrim.        float64
Cresc. Rec.5a           float64
Perfil_Risco_Cluster      int64
dtype: object


In [20]:
# --- O TRATAMENTO (Deixando TOP para o PBI) ---

# A. Garantir que o Ticker (Papel) seja texto puro (String)
df['Papel'] = df['Papel'].astype(str)

# B. Forçar conversão de todas as colunas de indicadores para números (Float)
# O Fundamentus às vezes traz valores como nulo ou caracteres estranhos.
# A função to_numeric com errors='coerce' força a virar número e o que for lixo vira vazio (NaN)
colunas_para_ignorar = ['Papel', 'Perfil_Risco_Cluster']
colunas_para_numeros = [col for col in df.columns if col not in colunas_para_ignorar]

for col in colunas_para_numeros:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# C. DICA DE OURO: Transformar a coluna de Cluster em Texto (Categoria)
# Se você deixar como 0, 1 e 2 (número inteiro), o Power BI vai tentar SOMAR isso, o que não faz sentido.
# Já vamos entregar o nome do grupo mastigado!
df['Perfil_Risco_Cluster'] = df['Perfil_Risco_Cluster'].map({
    0: '0 - Conservador / Vacas Leiteiras', 
    1: '1 - Crescimento / Valuation Justo', 
    2: '2 - Alto Risco / Voláteis'
})
# Nota: Os nomes dos perfis dependem da análise do seu K-Means, ajuste conforme a sua lógica!

# --- O RESULTADO ---
print("\n✨ TIPOS DE DADOS DEPOIS DA LIMPEZA:")
print(df.dtypes)
print("-" * 50)

# 4. Salvar de volta no banco, substituindo a tabela antiga pelos dados tratados
df.to_sql('ativos_clusterizados', con=con, if_exists='replace', index=False)

print("\nTabela atualizada e salva com sucesso! Pode conectar o Power BI.")


✨ TIPOS DE DADOS DEPOIS DA LIMPEZA:
Papel                       str
Cotação                 float64
P/L                     float64
P/VP                    float64
PSR                     float64
Div.Yield               float64
P/Ativo                 float64
P/Cap.Giro              float64
P/EBIT                  float64
P/Ativ Circ.Liq         float64
EV/EBIT                 float64
EV/EBITDA               float64
Mrg Bruta               float64
Mrg Ebit                float64
Mrg. Líq.               float64
Liq. Corr.              float64
ROIC                    float64
ROE                     float64
Liq.2meses              float64
Patrim. Líq             float64
Dív.Líq/ Patrim.        float64
Cresc. Rec.5a           float64
Perfil_Risco_Cluster        str
dtype: object
--------------------------------------------------

Tabela atualizada e salva com sucesso! Pode conectar o Power BI.
